# 02 · Calibration of N, U and the fraud threshold (§8.3)

Values before this calibration in `services/src/sofia_services/settings.py` (SIM): `DISPUTE_WINDOW_DAYS=90`,
`AMOUNT_THRESHOLD_USD=500`, `FRAUD_SCORE_THRESHOLD=0.8`. Data: silver from a local pipeline run
(`python -m sofia_data.pipeline --source local`), with gold identical to OPS's run.

Output: `analysis/results/policy_calibration.json`. The team sets the final values (`value`); this notebook holds the evidence.

In [ ]:
from common import connect, save_result, show

con = connect()

## 1. N: can a dispute be linked to its transaction?

`complaints` has no `transaction_id` (§14). We test (a) the affected product and (b) the claimed amount against the
customer's own transactions.

In [ ]:
con.sql("""CREATE OR REPLACE TEMP TABLE disputas AS
    SELECT complaint_id, customer_id, affected_product_id AS pid, creation_date::TIMESTAMP AS cdate,
           claimed_amount, currency
    FROM complaints WHERE subcategory IN ('Cargo no reconocido', 'Cobro indebido')""")
link = show(con, "Does the affected product belong to the complainant?", """
    SELECT count(*) AS disputas, count(d.pid) AS con_producto, count(p.product_id) AS producto_existe,
           count(*) FILTER (WHERE p.customer_id = d.customer_id) AS producto_del_cliente
    FROM disputas d LEFT JOIN products p ON p.product_id = d.pid""")
amount = show(con, "Amount match against the customer's own transactions (previous 365 days, ±1%)", """
    WITH l AS (
        SELECT d.complaint_id, d.claimed_amount, d.currency AS ccur, t.amount, t.currency AS tcur
        FROM disputas d JOIN transactions t ON t.customer_id = d.customer_id
         AND t.transaction_date::TIMESTAMP BETWEEN d.cdate - INTERVAL 365 DAY AND d.cdate)
    SELECT count(DISTINCT complaint_id) FILTER (WHERE claimed_amount IS NOT NULL) AS con_monto,
           count(DISTINCT complaint_id) FILTER (WHERE claimed_amount IS NOT NULL AND tcur = ccur
               AND abs(amount - claimed_amount) <= 0.01 * claimed_amount) AS con_match
    FROM l""")

**Result:** the affected product exists but never belongs to the complainant, and the amount match is at chance
level. The synthetic dataset does not link disputes to transactions, so N **cannot be calibrated from data**: it is
taken from regulation (below) and reported as a limitation. As an indirect reference, days from the customer's last
transaction to the complaint:

In [ ]:
lag = show(con, "Days from the customer's last transaction to the complaint (proxy, does not calibrate N)", """
    WITH l AS (
        SELECT d.complaint_id, min(date_diff('day', t.transaction_date::TIMESTAMP, d.cdate)) AS lag
        FROM disputas d JOIN transactions t ON t.customer_id = d.customer_id
         AND t.transaction_date::TIMESTAMP BETWEEN d.cdate - INTERVAL 365 DAY AND d.cdate
        GROUP BY 1)
    SELECT count(*) AS n, quantile_cont(lag, [0.5, 0.75, 0.9, 0.95]) AS p50_75_90_95 FROM l""")

## 2. U: amounts in USD

`amount_usd` is empty on every USD transaction (they are already in USD): the USD amount is
`coalesce(amount_usd, amount if currency = 'USD')`. This explains most of the "57% null" warning.

In [ ]:
usd = "coalesce(amount_usd, CASE WHEN currency = 'USD' THEN amount END)::DOUBLE"
purchases = show(con, "Approved purchases in USD", f"""
    SELECT count(*) AS n, quantile_cont({usd}, [0.5, 0.75, 0.8, 0.9, 0.95]) AS p50_75_80_90_95,
           round(avg(({usd} > 250)::INT), 3) AS share_sobre_250, round(avg(({usd} > 500)::INT), 3) AS share_sobre_500
    FROM transactions
    WHERE transaction_status = 'Approved' AND transaction_type = 'Purchase' AND {usd} IS NOT NULL""")  # noqa: S608

In [ ]:
claimed = show(con, "Claimed amount in disputes, in USD (daily exchange rate)", """
    WITH c AS (
        SELECT c.claimed_amount * CASE WHEN c.currency = 'USD' THEN 1 ELSE f.exchange_rate END AS usd
        FROM complaints c
        LEFT JOIN daily_exchange_rates f
          ON f.source_currency = c.currency AND f.target_currency = 'USD' AND f.date = c.creation_date::DATE
        WHERE c.subcategory IN ('Cargo no reconocido', 'Cobro indebido') AND c.claimed_amount IS NOT NULL)
    SELECT count(usd) AS n, quantile_cont(usd, [0.5, 0.75, 0.8, 0.9, 0.95]) AS p50_75_80_90_95,
           round(avg((usd > 500)::INT), 3) AS share_sobre_500
    FROM c""")

**Result:** synthetic purchases top out near 500 USD, so with U=500 no purchase goes to a human because of its
amount; on claimed amounts, U=500 sends ~23% to a human. U=250 splits purchases in half.

## 3. Fraud threshold

`fraud_score` runs from **0 to 100**, not from 0 to 1. The previous threshold (0.8) would flag ~97% of transactions as
suspicious as soon as bank-api reads gold.

In [ ]:
fraud = show(con, "Precision / recall by threshold (0–100 scale)", """
    SELECT th, round(sum((fraud_score >= th AND is_fraud)::INT) / nullif(sum((fraud_score >= th)::INT), 0), 3) AS prec,
           round(sum((fraud_score >= th AND is_fraud)::INT) / sum(is_fraud::INT), 3) AS rec,
           round(100 * avg((fraud_score >= th)::INT), 2) AS pct_marcada
    FROM transactions, (VALUES (0.8), (30), (40), (50), (70), (80)) v(th)
    WHERE fraud_score IS NOT NULL AND is_fraud IS NOT NULL GROUP BY th ORDER BY th""")

### Threshold by expected cost

No regulation sets the threshold: the practice is to minimize `cost = c_FN · undetected frauds + c_FP · extra reviews`,
with a missed fraud ≈ 40–50× an extra review (cost-sensitive fraud detection literature).

In [ ]:
cost = show(con, "Expected cost by threshold (0–100 scale)", """
    WITH g AS (
        SELECT th, sum((fraud_score >= th AND is_fraud)::INT) AS tp,
               sum((fraud_score >= th AND NOT is_fraud)::INT) AS fp,
               sum((fraud_score < th AND is_fraud)::INT) AS fn
        FROM transactions, (SELECT unnest(range(20, 46, 2)) AS th)
        WHERE fraud_score IS NOT NULL AND is_fraud IS NOT NULL GROUP BY th)
    SELECT th, tp, fp, fn, round(tp / (tp + fp), 3) AS prec, round(tp / (tp + fn), 3) AS rec,
           50 * fn + fp AS costo_50_1, 40 * fn + fp AS costo_40_1
    FROM g ORDER BY th""")

**Result:** the minimum cost is at **30** with both ratios (recall 0.69, precision 0.80, 0.08% to review).
At 40, ~360 more frauds are missed to save 609 reviews. Note: legitimate transactions in the synthetic dataset score
≤ ~30, hence the cliff between 28 and 30; a real bank would recalibrate on its own data.
`is_fraud` is the dataset's label: a real bank does not have it at dispute time (limitation).

## Team decision (2026-10-04)

| Parameter | Value | Justification |
|---|---|---|
| **N** | 90 days | Not calibratable from data. Mexico: 90 calendar days to object to unrecognized charges (Condusef, LTOSF art. 23); within Visa/Mastercard's 120-day chargeback window. Argentina is stricter (30 days from the statement, Law 25.065 art. 26): per-country windows as a production step |
| **U** | 500 USD | No regulation sets an amount. It automates almost every purchase (p95 475 USD) and sends 23% of claimed amounts to a human |
| **Fraud threshold** | 30 in gold = 0.30 in the policy | Minimum expected cost (above). bank-api converts gold from 0–100 to 0–1 when loading it |

In [ ]:
save_result("policy_calibration", {
    "dispute_transaction_link": {"product": link.to_dicts()[0], "amount": amount.to_dicts()[0]},
    "N_days": {"value": 90, "current": 90,
               "basis": "MX 90 días naturales (Condusef/LTOSF art. 23); Visa/MC 120 días",
               "proxy_lag_days": lag.to_dicts()[0],
               "rationale": "Not calibratable: disputes do not link to transactions. Taken from regulation."},
    "U_usd": {"value": 500, "current": 500, "approved_purchases": purchases.to_dicts()[0],
              "claimed_amounts": claimed.to_dicts()[0]},
    "fraud_score_threshold": {"value": 30, "policy_value_unit_scale": 0.30, "previous": 0.8, "scale": "0-100",
                              "tradeoff": fraud.to_dicts(), "cost_curve": cost.to_dicts(),
                              "rationale": "minimum expected cost with c_FN/c_FP = 40–50"},
    "data_quality_notes": {"amount_usd": "empty on USD rows; use coalesce(amount_usd, amount if USD)"},
})